In [ ]:
!pip install nltk

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk
import string,time
import re

In [ ]:
document = """Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.0 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature.
1 Introduction
Recurrent neural networks, long short-term memory [12] and gated recurrent [7] neural networks
in particular, have been ﬁrmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 29, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [31, 21, 13].
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea. Ashish, with Illia, designed and implemented the ﬁrst Transformer models and
has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head
attention and the parameter-free position representation and became the other person involved in nearly every
detail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and
tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and
efﬁcient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating
our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.
31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
statesht, as a function of the previous hidden stateht−1 and the input for positiont. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
signiﬁcant improvements in computational efﬁciency through factorization tricks [18] and conditional
computation [26], while also improving model performance in case of the latter. The fundamental
constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms
are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for signiﬁcantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2 Background
The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU
[20], ByteNet [15] and ConvS2S [8], all of which use convolutional neural networks as basic building
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
textual entailment and learning task-independent sentence representations [4, 22, 23, 19].
End-to-end memory networks are based on a recurrent attention mechanism instead of sequence-
aligned recurrence and have been shown to perform well on simple-language question answering and
language modeling tasks [28].
To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [14, 15] and [8].
3 Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].
Here, the encoder maps an input sequence of symbol representations (x1,...,x n) to a sequence
of continuous representations z = (z1,...,z n). Given z, the decoder then generates an output
sequence (y1,...,y m) of symbols one element at a time. At each step the model is auto-regressive
[9], consuming the previously generated symbols as additional input when generating the next.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-
2
Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for positioni can depend only on the known outputs at positions less thani.
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1 Scaled Dot-Product Attention
We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of
queries and keys of dimensiondk, and values of dimensiondv. We compute the dot products of the
3
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
query with all keys, divide each by√dk, and apply a softmax function to obtain the weights on the
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
into a matrixQ. The keys and values are also packed together into matricesK andV . We compute
the matrix of outputs as:
Attention(Q,K,V ) = softmax(QK T
√dk
)V (1)
The two most commonly used attention functions are additive attention [2], and dot-product (multi-
plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor
of 1√dk
. Additive attention computes the compatibility function using a feed-forward network with
a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is
much faster and more space-efﬁcient in practice, since it can be implemented using highly optimized
matrix multiplication code.
While for small values ofdk the two mechanisms perform similarly, additive attention outperforms
dot product attention without scaling for larger values ofdk [3]. We suspect that for large values of
dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has
extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk
.
3.2.2 Multi-Head Attention
Instead of performing a single attention function withdmodel-dimensional keys, values and queries,
we found it beneﬁcial to linearly project the queries, keys and valuesh times with different, learned
linear projections todk,dk anddv dimensions, respectively. On each of these projected versions of
queries, keys and values we then perform the attention function in parallel, yieldingdv-dimensional
output values. These are concatenated and once again projected, resulting in the ﬁnal values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
4To illustrate why the dot products get large, assume that the components of q and k are independent random
variables with mean 0 and variance 1. Then their dot product, q · k = ∑dk
i=1 qiki, has mean 0 and variance dk.
4
MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[31, 2, 8].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
information ﬂow in the decoder to preserve the auto-regressive property. We implement this
inside of scaled dot-product attention by masking out (setting to−∞) all values in the input
of the softmax which correspond to illegal connections. See Figure 2.
3.3 Position-wise Feed-Forward Networks
In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully
connected feed-forward network, which is applied to each position separately and identically. This
consists of two linear transformations with a ReLU activation in between.
FFN(x) = max(0,xW 1 +b1)W2 +b2 (2)
While the linear transformations are the same across different positions, they use different parameters
from layer to layer. Another way of describing this is as two convolutions with kernel size 1.
The dimensionality of input and output is dmodel = 512 , and the inner-layer has dimensionality
df f = 2048.
3.4 Embeddings and Softmax
Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimensiondmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embedding layers and the pre-softmax
linear transformation, similar to [24]. In the embedding layers, we multiply those weights by√dmodel.
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
5
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types.n is the sequence length,d is the representation dimension,k is the kernel
size of convolutions andr the size of the neighborhood in restricted self-attention.
Layer Type Complexity per Layer Sequential Maximum Path Length
Operations
Self-Attention O(n2·d) O(1) O(1)
Recurrent O(n·d2) O(n) O(n)
Convolutional O(k·n·d2) O(1) O(logk(n))
Self-Attention (restricted) O(r·n·d) O(1) O(n/r)
bottoms of the encoder and decoder stacks. The positional encodings have the same dimensiondmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings,
learned and ﬁxed [8].
In this work, we use sine and cosine functions of different frequencies:
PE (pos,2i) =sin(pos/100002i/dmodel)
PE (pos,2i+1) =cos(pos/100002i/dmodel)
wherepos is the position andi is the dimension. That is, each dimension of the positional encoding
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000· 2π. We
chose this function because we hypothesized it would allow the model to easily learn to attend by
relative positions, since for any ﬁxed offsetk,PE pos+k can be represented as a linear function of
PE pos.
We also experimented with using learned positional embeddings [8] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encountered
during training.
4 Why Self-Attention
In this section we compare various aspects of self-attention layers to the recurrent and convolu-
tional layers commonly used for mapping one variable-length sequence of symbol representations
(x1,...,x n) to another sequence of equal length (z1,...,z n), with xi,z i∈ Rd, such as a hidden
layer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we
consider three desiderata.
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations required.
The third is the path length between long-range dependencies in the network. Learning long-range
dependencies is a key challenge in many sequence transduction tasks. One key factor affecting the
ability to learn such dependencies is the length of the paths forward and backward signals have to
traverse in the network. The shorter these paths between any combination of positions in the input
and output sequences, the easier it is to learn long-range dependencies [11]. Hence we also compare
the maximum path length between any two input and output positions in networks composed of the
different layer types.
As noted in Table 1, a self-attention layer connects all positions with a constant number of sequentially
executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of
computational complexity, self-attention layers are faster than recurrent layers when the sequence
length n is smaller than the representation dimensionality d, which is most often the case with
sentence representations used by state-of-the-art models in machine translations, such as word-piece
[31] and byte-pair [25] representations. To improve computational performance for tasks involving
very long sequences, self-attention could be restricted to considering only a neighborhood of sizer in
6
the input sequence centered around the respective output position. This would increase the maximum
path length toO(n/r). We plan to investigate this approach further in future work.
A single convolutional layer with kernel widthk<n does not connect all pairs of input and output
positions. Doing so requires a stack ofO(n/k) convolutional layers in the case of contiguous kernels,
orO(logk(n)) in the case of dilated convolutions [ 15], increasing the length of the longest paths
between any two positions in the network. Convolutional layers are generally more expensive than
recurrent layers, by a factor of k. Separable convolutions [ 6], however, decrease the complexity
considerably, toO(k·n·d +n·d2). Even with k = n, however, the complexity of a separable
convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,
the approach we take in our model.
As side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions
from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic
and semantic structure of the sentences.
5 Training
This section describes the training regime for our models.
5.1 Training Data and Batching
We trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million
sentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-
target vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT
2014 English-French dataset consisting of 36M sentences and split tokens into a 32000 word-piece
vocabulary [31]. Sentence pairs were batched together by approximate sequence length. Each training
batch contained a set of sentence pairs containing approximately 25000 source tokens and 25000
target tokens.
5.2 Hardware and Schedule
We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using
the hyperparameters described throughout the paper, each training step took about 0.4 seconds. We
trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the
bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps
(3.5 days).
5.3 Optimizer
We used the Adam optimizer [17] withβ1 = 0.9,β2 = 0.98 andϵ = 10−9. We varied the learning
rate over the course of training, according to the formula:
lrate =d−0.5
model· min(step_num−0.5,step _num·warmup_steps−1.5) (3)
This corresponds to increasing the learning rate linearly for the ﬁrstwarmup_steps training steps,
and decreasing it thereafter proportionally to the inverse square root of the step number. We used
warmup_steps = 4000.
5.4 Regularization
We employ three types of regularization during training:
Residual Dropout We apply dropout [27] to the output of each sub-layer, before it is added to the
sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the
positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of
Pdrop = 0.1.
7
Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model
BLEU Training Cost (FLOPs)
EN-DE EN-FR EN-DE EN-FR
ByteNet [15] 23.75
Deep-Att + PosUnk [32] 39.2 1.0· 1020
GNMT + RL [31] 24.6 39.92 2.3· 1019 1.4· 1020
ConvS2S [8] 25.16 40.46 9.6· 1018 1.5· 1020
MoE [26] 26.03 40.56 2.0· 1019 1.2· 1020
Deep-Att + PosUnk Ensemble [32] 40.4 8.0· 1020
GNMT + RL Ensemble [31] 26.30 41.16 1.8· 1020 1.1· 1021
ConvS2S Ensemble [8] 26.36 41.29 7.7· 1019 1.2· 1021
Transformer (base model) 27.3 38.1 3.3 · 1018
Transformer (big) 28.4 41.0 2.3· 1019
Label Smoothing During training, we employed label smoothing of value ϵls = 0.1 [30]. This
hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score.
6 Results
6.1 Machine Translation
On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big)
in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0
BLEU, establishing a new state-of-the-art BLEU score of 28.4. The conﬁguration of this model is
listed in the bottom line of Table 3. Training took 3.5 days on 8 P100 GPUs. Even our base model
surpasses all previously published models and ensembles, at a fraction of the training cost of any of
the competitive models.
On the WMT 2014 English-to-French translation task, our big model achieves a BLEU score of 41.0,
outperforming all of the previously published single models, at less than 1/4 the training cost of the
previous state-of-the-art model. The Transformer (big) model trained for English-to-French used
dropout ratePdrop = 0.1, instead of 0.3.
For the base models, we used a single model obtained by averaging the last 5 checkpoints, which
were written at 10-minute intervals. For the big models, we averaged the last 20 checkpoints. We
used beam search with a beam size of 4 and length penaltyα = 0.6 [31]. These hyperparameters
were chosen after experimentation on the development set. We set the maximum output length during
inference to input length + 50, but terminate early when possible [31].
Table 2 summarizes our results and compares our translation quality and training costs to other model
architectures from the literature. We estimate the number of ﬂoating point operations used to train a
model by multiplying the training time, the number of GPUs used, and an estimate of the sustained
single-precision ﬂoating-point capacity of each GPU 5.
6.2 Model Variations
To evaluate the importance of different components of the Transformer, we varied our base model
in different ways, measuring the change in performance on English-to-German translation on the
development set, newstest2013. We used beam search as described in the previous section, but no
checkpoint averaging. We present these results in Table 3.
In Table 3 rows (A), we vary the number of attention heads and the attention key and value dimensions,
keeping the amount of computation constant, as described in Section 3.2.2. While single-head
attention is 0.9 BLEU worse than the best setting, quality also drops off with too many heads.
5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectively.
8
Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to
per-word perplexities.
N d model dff h d k dv Pdrop ϵls
train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36
4 5.19 25.3 50
8 4.88 25.5 80
256 32 32 5.75 24.5 28
1024 128 128 4.66 26.0 168
1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility is not easy and that a more sophisticated compatibility
function than dot product may be beneﬁcial. We further observe in rows (C) and (D) that, as expected,
bigger models are better, and dropout is very helpful in avoiding over-ﬁtting. In row (E) we replace our
sinusoidal positional encoding with learned positional embeddings [8], and observe nearly identical
results to the base model.
7 Conclusion
In this work, we presented the Transformer, the ﬁrst sequence transduction model based entirely on
attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with
multi-headed self-attention.
For translation tasks, the Transformer can be trained signiﬁcantly faster than architectures based
on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014
English-to-French translation tasks, we achieve a new state of the art. In the former task our best
model outperforms even all previously reported ensembles.
We are excited about the future of attention-based models and plan to apply them to other tasks. We
plan to extend the Transformer to problems involving input and output modalities other than text and
to investigate local, restricted attention mechanisms to efﬁciently handle large inputs and outputs
such as images, audio and video. Making generation less sequential is another research goals of ours.
The code we used to train and evaluate our models is available at https://github.com/
tensorflow/tensor2tensor.
Acknowledgements We are grateful to Nal Kalchbrenner and Stephan Gouws for their fruitful
comments, corrections and inspiration.
9
References
[1] Jimmy Lei Ba, Jamie Ryan Kiros, and Geoffrey E Hinton. Layer normalization. arXiv preprint
arXiv:1607.06450, 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly
learning to align and translate. CoRR, abs/1409.0473, 2014.
[3] Denny Britz, Anna Goldie, Minh-Thang Luong, and Quoc V . Le. Massive exploration of neural
machine translation architectures. CoRR, abs/1703.03906, 2017.
[4] Jianpeng Cheng, Li Dong, and Mirella Lapata. Long short-term memory-networks for machine
reading. arXiv preprint arXiv:1601.06733, 2016.
[5] Kyunghyun Cho, Bart van Merrienboer, Caglar Gulcehre, Fethi Bougares, Holger Schwenk,
and Yoshua Bengio. Learning phrase representations using rnn encoder-decoder for statistical
machine translation. CoRR, abs/1406.1078, 2014.
[6] Francois Chollet. Xception: Deep learning with depthwise separable convolutions. arXiv
preprint arXiv:1610.02357, 2016.
[7] Junyoung Chung, Çaglar Gülçehre, Kyunghyun Cho, and Yoshua Bengio. Empirical evaluation
of gated recurrent neural networks on sequence modeling. CoRR, abs/1412.3555, 2014.
[8] Jonas Gehring, Michael Auli, David Grangier, Denis Yarats, and Yann N. Dauphin. Convolu-
tional sequence to sequence learning. arXiv preprint arXiv:1705.03122v2, 2017.
[9] Alex Graves. Generating sequences with recurrent neural networks. arXiv preprint
arXiv:1308.0850, 2013.
[10] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for im-
age recognition. In Proceedings of the IEEE Conference on Computer Vision and Pattern
Recognition, pages 770–778, 2016.
[11] Sepp Hochreiter, Yoshua Bengio, Paolo Frasconi, and Jürgen Schmidhuber. Gradient ﬂow in
recurrent nets: the difﬁculty of learning long-term dependencies, 2001.
[12] Sepp Hochreiter and Jürgen Schmidhuber. Long short-term memory. Neural computation,
9(8):1735–1780, 1997.
[13] Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu. Exploring
the limits of language modeling. arXiv preprint arXiv:1602.02410, 2016.
[14] Łukasz Kaiser and Ilya Sutskever. Neural GPUs learn algorithms. In International Conference
on Learning Representations (ICLR), 2016.
[15] Nal Kalchbrenner, Lasse Espeholt, Karen Simonyan, Aaron van den Oord, Alex Graves, and Ko-
ray Kavukcuoglu. Neural machine translation in linear time.arXiv preprint arXiv:1610.10099v2,
2017.
[16] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks.
In International Conference on Learning Representations , 2017.
[17] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015.
[18] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint
arXiv:1703.10722, 2017.
[19] Zhouhan Lin, Minwei Feng, Cicero Nogueira dos Santos, Mo Yu, Bing Xiang, Bowen
Zhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint
arXiv:1703.03130, 2017.
[20] Samy Bengio Łukasz Kaiser. Can active memory replace attention? In Advances in Neural
Information Processing Systems, (NIPS), 2016.
10
[21] Minh-Thang Luong, Hieu Pham, and Christopher D Manning. Effective approaches to attention-
based neural machine translation. arXiv preprint arXiv:1508.04025, 2015.
[22] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention
model. In Empirical Methods in Natural Language Processing , 2016.
[23] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive
summarization. arXiv preprint arXiv:1705.04304, 2017.
[24] Oﬁr Press and Lior Wolf. Using the output embedding to improve language models. arXiv
preprint arXiv:1608.05859, 2016.
[25] Rico Sennrich, Barry Haddow, and Alexandra Birch. Neural machine translation of rare words
with subword units. arXiv preprint arXiv:1508.07909, 2015.
[26] Noam Shazeer, Azalia Mirhoseini, Krzysztof Maziarz, Andy Davis, Quoc Le, Geoffrey Hinton,
and Jeff Dean. Outrageously large neural networks: The sparsely-gated mixture-of-experts
layer. arXiv preprint arXiv:1701.06538, 2017.
[27] Nitish Srivastava, Geoffrey E Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdi-
nov. Dropout: a simple way to prevent neural networks from overﬁtting. Journal of Machine
Learning Research, 15(1):1929–1958, 2014.
[28] Sainbayar Sukhbaatar, arthur szlam, Jason Weston, and Rob Fergus. End-to-end memory
networks. In C. Cortes, N. D. Lawrence, D. D. Lee, M. Sugiyama, and R. Garnett, editors,
Advances in Neural Information Processing Systems 28 , pages 2440–2448. Curran Associates,
Inc., 2015.
[29] Ilya Sutskever, Oriol Vinyals, and Quoc VV Le. Sequence to sequence learning with neural
networks. In Advances in Neural Information Processing Systems , pages 3104–3112, 2014.
[30] Christian Szegedy, Vincent Vanhoucke, Sergey Ioffe, Jonathon Shlens, and Zbigniew Wojna.
Rethinking the inception architecture for computer vision. CoRR, abs/1512.00567, 2015.
[31] Yonghui Wu, Mike Schuster, Zhifeng Chen, Quoc V Le, Mohammad Norouzi, Wolfgang
Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Google’s neural machine
translation system: Bridging the gap between human and machine translation. arXiv preprint
arXiv:1609.08144, 2016.
[32] Jie Zhou, Ying Cao, Xuguang Wang, Peng Li, and Wei Xu. Deep recurrent models with
fast-forward connections for neural machine translation. CoRR, abs/1606.04199, 2016.
11
"""

In [ ]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import re
import string
from nltk.tokenize import word_tokenize

def word_tokenization(document):
    exclude = '!"#$%&\'()*+,-./:;<=>?[\\]^_`{|}~'

    for char in exclude:
        document = document.replace(char, '')

    document = re.sub(r"[∗†]", "", document)

    tokens = word_tokenize(document.lower())

    return tokens

In [ ]:
tokens = word_tokenization(document)
tokens

['attention',
 'is',
 'all',
 'you',
 'need',
 'ashish',
 'vaswani',
 'google',
 'brain',
 'avaswani',
 '@',
 'googlecom',
 'noam',
 'shazeer',
 'google',
 'brain',
 'noam',
 '@',
 'googlecom',
 'niki',
 'parmar',
 'google',
 'research',
 'nikip',
 '@',
 'googlecom',
 'jakob',
 'uszkoreit',
 'google',
 'research',
 'usz',
 '@',
 'googlecom',
 'llion',
 'jones',
 'google',
 'research',
 'llion',
 '@',
 'googlecom',
 'aidan',
 'n',
 'gomez',
 'university',
 'of',
 'toronto',
 'aidan',
 '@',
 'cstorontoedu',
 'łukasz',
 'kaiser',
 'google',
 'brain',
 'lukaszkaiser',
 '@',
 'googlecom',
 'illia',
 'polosukhin‡',
 'illiapolosukhin',
 '@',
 'gmailcom',
 'abstract',
 'the',
 'dominant',
 'sequence',
 'transduction',
 'models',
 'are',
 'based',
 'on',
 'complex',
 'recurrent',
 'or',
 'convolutional',
 'neural',
 'networks',
 'that',
 'include',
 'an',
 'encoder',
 'and',
 'a',
 'decoder',
 'the',
 'best',
 'performing',
 'models',
 'also',
 'connect',
 'the',
 'encoder',
 'and',
 'decoder',

In [ ]:
# build vocab
def build_vocab(tokens):
  vocab = {'<unk>':0}
  for token in Counter(tokens).keys():
    if token not in vocab:
      vocab[token] = len(vocab)
  return vocab

In [ ]:
vocab = build_vocab(tokens)
vocab

{'<unk>': 0,
 'attention': 1,
 'is': 2,
 'all': 3,
 'you': 4,
 'need': 5,
 'ashish': 6,
 'vaswani': 7,
 'google': 8,
 'brain': 9,
 'avaswani': 10,
 '@': 11,
 'googlecom': 12,
 'noam': 13,
 'shazeer': 14,
 'niki': 15,
 'parmar': 16,
 'research': 17,
 'nikip': 18,
 'jakob': 19,
 'uszkoreit': 20,
 'usz': 21,
 'llion': 22,
 'jones': 23,
 'aidan': 24,
 'n': 25,
 'gomez': 26,
 'university': 27,
 'of': 28,
 'toronto': 29,
 'cstorontoedu': 30,
 'łukasz': 31,
 'kaiser': 32,
 'lukaszkaiser': 33,
 'illia': 34,
 'polosukhin‡': 35,
 'illiapolosukhin': 36,
 'gmailcom': 37,
 'abstract': 38,
 'the': 39,
 'dominant': 40,
 'sequence': 41,
 'transduction': 42,
 'models': 43,
 'are': 44,
 'based': 45,
 'on': 46,
 'complex': 47,
 'recurrent': 48,
 'or': 49,
 'convolutional': 50,
 'neural': 51,
 'networks': 52,
 'that': 53,
 'include': 54,
 'an': 55,
 'encoder': 56,
 'and': 57,
 'a': 58,
 'decoder': 59,
 'best': 60,
 'performing': 61,
 'also': 62,
 'connect': 63,
 'through': 64,
 'mechanism': 65,
 'we': 66,

In [ ]:
len(vocab)

1512

In [ ]:
# extract sentences from data
def sentence_tokenization(document):
  input_sentences = document.split('\n')
  return input_sentences


In [ ]:
# extract sentences from data
input_sentences = sentence_tokenization(document)
input_sentences

['Attention Is All You Need',
 'Ashish Vaswani∗',
 'Google Brain',
 'avaswani@google.com',
 'Noam Shazeer∗',
 'Google Brain',
 'noam@google.com',
 'Niki Parmar∗',
 'Google Research',
 'nikip@google.com',
 'Jakob Uszkoreit∗',
 'Google Research',
 'usz@google.com',
 'Llion Jones∗',
 'Google Research',
 'llion@google.com',
 'Aidan N. Gomez∗†',
 'University of Toronto',
 'aidan@cs.toronto.edu',
 'Łukasz Kaiser∗',
 'Google Brain',
 'lukaszkaiser@google.com',
 'Illia Polosukhin∗‡',
 'illia.polosukhin@gmail.com',
 'Abstract',
 'The dominant sequence transduction models are based on complex recurrent or',
 'convolutional neural networks that include an encoder and a decoder. The best',
 'performing models also connect the encoder and decoder through an attention',
 'mechanism. We propose a new simple network architecture, the Transformer,',
 'based solely on attention mechanisms, dispensing with recurrence and convolutions',
 'entirely. Experiments on two machine translation tasks show these m

In [ ]:
def text_to_indices(sentence, vocab):
  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])
  return numerical_sentence

In [ ]:
input_numerical_sentences = []
for sentence in input_sentences:
   input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

In [ ]:
input_numerical_sentences

[[1, 2, 3, 4, 5],
 [6, 0],
 [8, 9],
 [10, 11, 0],
 [13, 0],
 [8, 9],
 [13, 11, 0],
 [15, 0],
 [8, 17],
 [18, 11, 0],
 [19, 0],
 [8, 17],
 [21, 11, 0],
 [22, 0],
 [8, 17],
 [22, 11, 0],
 [24, 0, 0],
 [27, 28, 29],
 [24, 11, 0],
 [31, 0],
 [8, 9],
 [33, 11, 0],
 [34, 0],
 [0, 11, 0],
 [38],
 [39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
 [50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 0, 39, 60],
 [61, 43, 62, 63, 39, 56, 57, 59, 64, 55, 1],
 [65, 0, 66, 67, 58, 68, 69, 70, 71, 0, 39, 72, 0],
 [45, 73, 46, 1, 74, 0, 75, 76, 77, 57, 78],
 [79, 0, 80, 46, 81, 82, 83, 84, 85, 86, 43, 87],
 [88, 89, 90, 91, 92, 93, 94, 95, 57, 96, 97],
 [98, 99, 87, 100, 0, 101, 102, 103, 0, 105, 46, 39, 106, 107, 0],
 [0, 83, 110, 0, 111, 112, 39, 113, 60, 114, 0, 115],
 [116, 0, 117, 112, 118, 105, 0, 46, 39, 106, 107, 0, 83, 110, 0],
 [101, 102, 120, 58, 68, 0, 0, 105, 123, 28, 0, 125],
 [126, 127, 0, 129, 46, 130, 131, 0, 58, 132, 133, 28, 39, 126, 134, 28, 39],
 [60, 43, 135, 39, 136, 0],
 [137, 138],
 [48, 51

In [ ]:
# training sequences
def training_sequences(input_numerical_sentences):
  training_sequence = []
  for sentence in input_numerical_sentences:
    for i in range(1, len(sentence)):
      training_sequences.append(sentence[:i+1])
  return training_sequence

In [ ]:
training_sequence = training_sequences(input_numerical_sentences)
training_sequence

[[1, 2],
 [1, 2, 3],
 [1, 2, 3, 4],
 [1, 2, 3, 4, 5],
 [6, 0],
 [8, 9],
 [10, 11],
 [10, 11, 0],
 [13, 0],
 [8, 9],
 [13, 11],
 [13, 11, 0],
 [15, 0],
 [8, 17],
 [18, 11],
 [18, 11, 0],
 [19, 0],
 [8, 17],
 [21, 11],
 [21, 11, 0],
 [22, 0],
 [8, 17],
 [22, 11],
 [22, 11, 0],
 [24, 0],
 [24, 0, 0],
 [27, 28],
 [27, 28, 29],
 [24, 11],
 [24, 11, 0],
 [31, 0],
 [8, 9],
 [33, 11],
 [33, 11, 0],
 [34, 0],
 [0, 11],
 [0, 11, 0],
 [39, 40],
 [39, 40, 41],
 [39, 40, 41, 42],
 [39, 40, 41, 42, 43],
 [39, 40, 41, 42, 43, 44],
 [39, 40, 41, 42, 43, 44, 45],
 [39, 40, 41, 42, 43, 44, 45, 46],
 [39, 40, 41, 42, 43, 44, 45, 46, 47],
 [39, 40, 41, 42, 43, 44, 45, 46, 47, 48],
 [39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
 [50, 51],
 [50, 51, 52],
 [50, 51, 52, 53],
 [50, 51, 52, 53, 54],
 [50, 51, 52, 53, 54, 55],
 [50, 51, 52, 53, 54, 55, 56],
 [50, 51, 52, 53, 54, 55, 56, 57],
 [50, 51, 52, 53, 54, 55, 56, 57, 58],
 [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
 [50, 51, 52, 53, 54, 55, 56, 57, 58, 59

In [ ]:
# padding

def padding(training_sequence):
  len_list = []
  for sequence in training_sequence:
    len_list.append(len(sequence))

  padded_training_sequence = []
  for sequence in training_sequence:
    padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

  # converting the 2D list into tensor
  padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)
  return padded_training_sequence

In [ ]:
padded_training_sequence = padding(training_sequences)
padded_training_sequence.shape

torch.Size([5490, 34])

In [ ]:
# extracting X, y
def extracting_X_y(padded_training_sequence):
  X = padded_training_sequence[:, :-1]
  y = padded_training_sequence[:, -1]

  return X,y

In [ ]:
X,y = extracting_X_y(padded_training_sequence)

In [ ]:
X

tensor([[   0,    0,    0,  ...,    0,    0,    1],
        [   0,    0,    0,  ...,    0,    1,    2],
        [   0,    0,    0,  ...,    1,    2,    3],
        ...,
        [   0,    0,    0,  ..., 1229,    0,    0],
        [   0,    0,    0,  ...,    0,    0,    0],
        [   0,    0,    0,  ...,    0,    0, 1220]])

In [ ]:
y

tensor([   2,    3,    4,  ...,    0, 1220,    0])

In [ ]:
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self,idx):
    return self.X[idx], self.y[idx]

In [ ]:
dataset = CustomDataset(X,y)

In [ ]:
def dataLoader(dataset):
  dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
  for input, output in dataloader:
      input, output = input.to(device), output.to(device)
      input, output = input.long(), output.long()
  return input, output


In [ ]:
len(dataset)

5490

In [ ]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 1]),
 tensor(2))

In [ ]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
for input, output in dataloader:
  print(input, output)

tensor([[   0,    0,    0,  ...,   57,  200,   39],
        [   0,    0,    0,  ...,   58,  752,   28],
        [   0,    0,    0,  ...,    0,  367,  153],
        ...,
        [   0,    0,    0,  ..., 1407, 1408,    0],
        [   0,    0,    0,  ...,  235,  680,   90],
        [   0,    0,    0,  ...,    0,   39,   72]]) tensor([ 201,  856,   87, 1121,   28,  183,    0,  392,    0,  640,    0,   58,
         695,  851,    0,  304,    0,    0,   39,   87,    0,    0,    0,  266,
           2,  674,  744,  395,    0,   58,   39,    2])
tensor([[   0,    0,    0,  ...,  241,    0,  242],
        [   0,    0,    0,  ...,    0,  640,    0],
        [   0,    0,    0,  ...,   53,   39,  250],
        ...,
        [   0,    0,    0,  ...,    0,    0,   90],
        [   0,    0,    0,  ...,    0,    0,   59],
        [   0,    0,    0,  ...,    0,    0, 1388]]) tensor([   0,    0,  503,    0,    0, 1216,  383,   41, 1069,   57,    0,  159,
         689,    0,    0,    0,   56,  115,   62,  

In [ ]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(1565, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [ ]:
model = LSTMModel(len(vocab))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model.to(device)

LSTMModel(
  (embedding): Embedding(1565, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=1512, bias=True)
)

In [ ]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    batch_x, batch_y = batch_x.long(), batch_y.long()

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 966.4308
Epoch: 2, Loss: 856.8152
Epoch: 3, Loss: 805.7176
Epoch: 4, Loss: 752.7648
Epoch: 5, Loss: 699.7995
Epoch: 6, Loss: 646.4293
Epoch: 7, Loss: 594.6633
Epoch: 8, Loss: 543.3932
Epoch: 9, Loss: 494.0008
Epoch: 10, Loss: 447.2416
Epoch: 11, Loss: 402.6226
Epoch: 12, Loss: 361.3044
Epoch: 13, Loss: 323.2936
Epoch: 14, Loss: 288.3244
Epoch: 15, Loss: 258.3083
Epoch: 16, Loss: 230.1335
Epoch: 17, Loss: 205.5407
Epoch: 18, Loss: 184.5544
Epoch: 19, Loss: 165.6118
Epoch: 20, Loss: 148.6357
Epoch: 21, Loss: 133.7420
Epoch: 22, Loss: 120.3207
Epoch: 23, Loss: 108.9084
Epoch: 24, Loss: 99.3709
Epoch: 25, Loss: 90.1851
Epoch: 26, Loss: 82.6015
Epoch: 27, Loss: 75.9241
Epoch: 28, Loss: 70.4170
Epoch: 29, Loss: 65.4949
Epoch: 30, Loss: 61.0392
Epoch: 31, Loss: 57.5926


In [ ]:
# prediction
def prediction(model, vocab, text):

  # tokenize

  tokenized_text = word_tokenize(text.lower())
  #print(tokenized_text)

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)
  #print(numerical_text)

  # padding
  padded_text = torch.tensor([0] *(61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)
  #print(padded_text)

  # send to model
  output = model(padded_text)
  #print(output.shape)
  #print(output)

  # predicted index
  value, index = torch.max(output,dim=1)


  # merge with text
  return text + " " + list(vocab.keys())[index]


In [ ]:
prediction(model, vocab, "The dominant sequence transduction models")

In [ ]:
num_tokens = 10
input_text = "The dominant sequence transduction models"
for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "epoch": epoch
}, "model.pth")

In [ ]:
vocab = build_vocab(tokens)

import torch
torch.save(vocab, "vocab.pth")